# Prédire la Volatilité Réalisée — Chemin de recherche

**Auteur :** Paul MONTIER — Mars 2026

---

**Objectif :** Construire un modèle capable de prédire, chaque jour, quelle sera la volatilité réalisée d'un titre sur les 5 prochains jours.

Ce notebook retrace le **cheminement de pensée** de la recherche — pas un changelog technique, mais le raisonnement à chaque étape : pourquoi telle idée, quel résultat, quelle surprise, quelle leçon.

**Fil conducteur :**
1. On part des données et on comprend ce qu'on cherche à prédire
2. On construit un premier modèle → résultats trop beaux pour être vrais
3. On enquête → 3 biais méthodologiques expliquent 52% du signal apparent
4. On repart d'une base honnête, puis on améliore progressivement
5. On découvre un résultat surprenant sur les interactions non linéaires

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'font.family': 'serif', 'font.size': 10, 'axes.labelsize': 11,
    'axes.titlesize': 12, 'figure.figsize': (10, 5), 'figure.dpi': 120,
})

from har_rv_model import (
    HARRVModel, adaptive_train_window, build_cross_sectional_features,
    ADAPTIVE_WINDOW_ANCHORS, DATA_DIR
)

print('Imports OK')

---
## 1. Comprendre ce qu'on cherche à prédire

La **volatilité** d'un titre financier mesure l'amplitude de ses fluctuations de prix. Elle est centrale en finance : un gestionnaire en a besoin pour calibrer ses positions, un trader d'options pour estimer si la vol implicite est sur- ou sous-évaluée, un risk manager pour anticiper les régimes de stress.

Contrairement aux rendements (dont la prédictibilité est débattue), la volatilité est **nettement plus prévisible** grâce à trois propriétés empiriques :
- **Persistance** : la vol élevée tend à rester élevée
- **Clustering** : les périodes de forte vol se regroupent
- **Mémoire longue** : les chocs mettent des semaines à se dissiper

Notre mesure est la **volatilité réalisée (RV)** : au lieu d'utiliser un seul rendement journalier, on agrège ~350 rendements 1-minute par jour.

In [ ]:
STOCKS = ['AAPL', 'MSFT', 'NVDA', 'JPM', 'META']

model = HARRVModel(model_type='xgboost')

# Charger les données
all_data = {}
for symbol in STOCKS:
    df = model.get_stock_data(symbol)
    if df is not None:
        all_data[symbol] = df
        print(f"{symbol}: {len(df)} jours, "
              f"{df.index.min().date()} → {df.index.max().date()}")

vix = model.get_vix()
print(f"\nVIX: {len(vix)} jours, range [{vix.min():.1f}, {vix.max():.1f}]")

In [ ]:
# La volatilité réalisée intraday sur nos 5 titres
fig, axes = plt.subplots(len(STOCKS), 1, figsize=(12, 3 * len(STOCKS)), sharex=True)

for ax, symbol in zip(axes, STOCKS):
    df = all_data[symbol]
    if 'rv_intraday' in df.columns:
        rv = df['rv_intraday'].dropna()
        ax.plot(rv.index, rv.values * 100, linewidth=0.5, alpha=0.7)
        rv_m = rv.rolling(22).mean() * 100
        ax.plot(rv_m.index, rv_m.values, linewidth=1.5, color='red', label='Moyenne 22j')
        ax.set_ylabel('RV (%)')
        ax.set_title(f'{symbol}')
        ax.legend(loc='upper right', fontsize=8)
        ax.set_ylim(0, rv.quantile(0.99) * 100 * 1.5)

fig.suptitle('Volatilité Réalisée Intraday (1-min) — on voit le clustering et la persistance',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Statistiques descriptives
stats = []
for symbol in STOCKS:
    df = all_data[symbol]
    rv = df['rv_intraday'] if 'rv_intraday' in df.columns else np.abs(df['returns'])
    stats.append({
        'Stock': symbol,
        'Jours': len(df),
        'RV moy (%)': f"{rv.mean()*100:.2f}",
        'RV std (%)': f"{rv.std()*100:.2f}",
        'Rend. moy (%)': f"{df['returns'].mean()*100:.4f}",
        'Rend. std (%)': f"{df['returns'].std()*100:.2f}",
    })

pd.DataFrame(stats).set_index('Stock')

**Observation clé :** NVDA est ~2× plus volatile que JPM. Les dynamiques sont très différentes selon le secteur. Un bon modèle doit fonctionner sur tous ces profils.

---
## 2. Le point de départ : le modèle HAR-RV

Le modèle HAR-RV de Corsi (2009) est le standard académique. Son idée est élégante : décomposer la volatilité passée en 3 échelles temporelles :

$$RV_{t+h} = \beta_0 + \beta_d \cdot RV_t^{(d)} + \beta_w \cdot RV_t^{(w)} + \beta_m \cdot RV_t^{(m)} + \varepsilon$$

- **Journalier** $RV^{(d)}$ : ce que font les day-traders
- **Hebdomadaire** $RV^{(w)}$ : l'horizon des gestionnaires de portefeuille
- **Mensuel** $RV^{(m)}$ : celui des institutionnels

C'est simple et ça marche... mais c'est **linéaire**. Or la volatilité a des comportements non linéaires : la réponse à un choc dépend du régime de marché (calme vs crise), et les features interagissent entre elles.

**Notre approche :** Garder la philosophie multi-échelle du HAR-RV, mais remplacer la régression linéaire par **XGBoost** (arbres de décision boostés) et étendre les features de 3 à 20.

### Les 20 features — chaque catégorie capture une dimension différente

| # | Catégorie | Features | Ce qu'elles capturent |
|---|-----------|----------|----------------------|
| 1-6 | Volatilité multi-échelle | RV_w, RV_m, RV_q, RV_neg_w, J_w, VIX | Cœur du HAR-RV + vol implicite |
| 7-9 | Multi-timeframe journalier | RV_overnight, Parkinson, Vol_ratio | Overnight, range intraday, activité |
| 10-13 | Microstructure 1-min | RV_1min, Vol_1m5m, Vol_AM_PM, Autocorr | Bruit de marché, flux d'info |
| 14-16 | Moments supérieurs | RSkew, RKurt, Jump_ratio | Forme de la distribution intraday |
| 17-18 | Effet de levier | Ret_w, Leverage_22d | Asymétrie rendement ↔ vol |
| 19-20 | Cross-sectionnel | RV_w_zscore, RV_w_rank_Δ | Position relative dans l'univers |

In [ ]:
# Construire toutes les features pour AAPL comme exemple
cs_features = build_cross_sectional_features(all_data)
df_aapl = all_data['AAPL']

features_aapl = model.create_features(df_aapl, vix, cs_features.get('AAPL'))
print(f"{features_aapl.shape[1]} features construites : {list(features_aapl.columns)}")
features_aapl.tail(3).round(4)

In [ ]:
# Matrice de corrélation — vérifier que les features ne sont pas redondantes
corr = features_aapl.dropna().corr()

fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(corr.columns, fontsize=8)
plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_title('Corrélations entre features (AAPL)')

for i in range(len(corr)):
    for j in range(len(corr)):
        if i != j and abs(corr.iloc[i, j]) > 0.7:
            ax.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center',
                    fontsize=6, fontweight='bold')

plt.tight_layout()
plt.show()

# Lister les paires fortement corrélées
print("Corrélations fortes (|ρ| > 0.7) :")
for i in range(len(corr)):
    for j in range(i+1, len(corr)):
        if abs(corr.iloc[i, j]) > 0.7:
            print(f"  {corr.columns[i]} ↔ {corr.columns[j]}: ρ = {corr.iloc[i, j]:.3f}")

### La variable cible

$$y_t = \log\left(\frac{\overline{RV}_{t+1:t+5}}{RV_t^{(m)}}\right)$$

C'est le log-ratio entre la volatilité future (5 jours) et la volatilité mensuelle courante. Le log stationnarise la cible, et le dénominateur $RV_m$ normalise par le niveau ambiant de volatilité.

In [ ]:
target_aapl = model.create_target(df_aapl)
valid = ~np.isnan(target_aapl)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(target_aapl[valid], bins=80, color='#3498db', edgecolor='white', alpha=0.8)
axes[0].axvline(0, color='red', linestyle='--', linewidth=1)
axes[0].set_xlabel('log(RV_future / RV_m)')
axes[0].set_ylabel('Fréquence')
axes[0].set_title('Distribution de la cible (AAPL)')
axes[0].text(0.05, 0.95, f'μ = {np.nanmean(target_aapl):.3f}\nσ = {np.nanstd(target_aapl):.3f}',
             transform=axes[0].transAxes, va='top', fontsize=9,
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

dates = df_aapl.index[valid]
axes[1].plot(dates, target_aapl[valid], linewidth=0.4, alpha=0.6)
axes[1].axhline(0, color='red', linestyle='--', linewidth=0.8)
axes[1].set_ylabel('log(RV_future / RV_m)')
axes[1].set_title('Cible dans le temps — centrée autour de 0, pics pendant les crises')

plt.tight_layout()
plt.show()

---
## 3. Premier résultat — et premier doute

Le premier modèle (XGBoost + 6 features de base + walk-forward) affichait :

> **IC = 0.61, Hit Rate = 71%**

En prédiction de volatilité, un IC > 0.30 est déjà considéré comme excellent. Un IC de 0.61 signifierait qu'on a trouvé un signal exceptionnel... ou qu'il y a un problème.

**Règle d'or en finance quantitative :** si vos résultats sont trop beaux, cherchez le bug avant de chercher le champagne.

L'investigation a révélé **3 biais cumulatifs** qui gonflaient artificiellement l'IC.

### Biais 1 — Chevauchement des targets

Notre cible est calculée sur un horizon de 5 jours. La cible du jour $t$ utilise $\{r_{t+1}, ..., r_{t+5}\}$, celle du jour $t-1$ utilise $\{r_t, ..., r_{t+4}\}$. **4 jours sur 5 se chevauchent.**

Si on entraîne le modèle jusqu'à $t-1$ (inclus), il a accès indirectement à 80% de l'information de la cible du jour $t$. Ce n'est pas du look-ahead au sens strict, mais l'effet est similaire.

In [ ]:
# Visualisation du chevauchement
fig, ax = plt.subplots(figsize=(10, 3))

labels = ['t-6', 't-5', 't-4', 't-3', 't-2', 't-1', 't', 't+1', 't+2', 't+3', 't+4', 't+5', 't+6', 't+7', 't+8']

for i in range(5, 10):
    ax.barh(1, 0.9, left=i+0.05, height=0.3, color='#3498db', alpha=0.7)
ax.text(7, 1, 'Cible du dernier jour train (t-1)', ha='center', va='center', fontsize=9, fontweight='bold')

for i in range(6, 11):
    ax.barh(0, 0.9, left=i+0.05, height=0.3, color='#e74c3c', alpha=0.7)
ax.text(8, 0, 'Cible du jour de test (t)', ha='center', va='center', fontsize=9, fontweight='bold')

for i in range(6, 10):
    ax.barh(0.5, 0.9, left=i+0.05, height=0.15, color='#f39c12', alpha=0.9)
ax.text(8, 0.5, '4 jours en commun (80%)', ha='center', va='center', fontsize=8, color='#d35400')

ax.set_yticks([0, 0.5, 1])
ax.set_yticklabels(['Test', 'Overlap', 'Train'])
ax.set_xticks(range(15))
ax.set_xticklabels(labels, fontsize=8)
ax.axvline(5.5, color='gray', linestyle='--', linewidth=1, label='Frontière sans purge')
ax.axvline(1.5, color='green', linestyle='--', linewidth=1, label='Frontière avec purge (gap=4)')
ax.set_title('Le problème du chevauchement des targets à horizon h=5')
ax.legend(fontsize=8, loc='lower right')
ax.set_xlim(-0.5, 14.5)

plt.tight_layout()
plt.show()

print("Correction : introduire un 'purge gap' de h-1 = 4 jours entre train et test.")
print("Impact : IC passe de 0.61 à ~0.45 (-26%)")

### Biais 2 — Couplage target-feature

La formulation initiale de la cible était :

$$y_t = \log\left(\frac{RV_{t+5}}{RV_w}\right) \quad \text{avec } RV_w \text{ comme dénominateur ET comme feature}$$

Le problème : si $RV_w$ est élevé un jour donné, la cible est **mécaniquement** basse (indépendamment de la vol future). Le modèle apprend cette corrélation triviale et obtient un bon IC... sans prédire quoi que ce soit d'utile.

In [ ]:
# Démonstration du couplage
df_demo = all_data['AAPL']
returns = df_demo['returns'].values

rv_w = pd.Series(np.abs(returns)).rolling(5).mean().values

# Cible biaisée (denom = RV_w)
rv_future = np.full(len(returns), np.nan)
for i in range(len(returns) - 5):
    rv_future[i] = np.std(returns[i+1:i+6], ddof=1)

with np.errstate(divide='ignore', invalid='ignore'):
    target_biased = np.where(
        (rv_w > 1e-12) & np.isfinite(rv_future) & (rv_future > 0),
        np.log(rv_future / rv_w), np.nan)

# Cible corrigée (denom = RV_m)
rv_m = pd.Series(np.abs(returns)).rolling(22).mean().values
with np.errstate(divide='ignore', invalid='ignore'):
    target_corrected = np.where(
        (rv_m > 1e-12) & np.isfinite(rv_future) & (rv_future > 0),
        np.log(rv_future / rv_m), np.nan)

valid = np.isfinite(rv_w) & np.isfinite(target_biased) & np.isfinite(target_corrected)
rho_biased, _ = spearmanr(rv_w[valid], target_biased[valid])
rho_corrected, _ = spearmanr(rv_w[valid], target_corrected[valid])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
s = valid & (np.arange(len(rv_w)) % 5 == 0)

axes[0].scatter(rv_w[s], target_biased[s], alpha=0.3, s=5, color='#e74c3c')
axes[0].set_xlabel('RV_w (feature)')
axes[0].set_ylabel('log(RV_future / RV_w)')
axes[0].set_title(f'Dénominateur = RV_w\nρ = {rho_biased:.3f} — corrélation mécanique !')

axes[1].scatter(rv_w[s], target_corrected[s], alpha=0.3, s=5, color='#27ae60')
axes[1].set_xlabel('RV_w (feature)')
axes[1].set_ylabel('log(RV_future / RV_m)')
axes[1].set_title(f'Dénominateur = RV_m (22j)\nρ = {rho_corrected:.3f} — découplé')

plt.suptitle('Biais 2 : le couplage target-feature crée un signal artificiel', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print(f"Correction : remplacer le dénominateur RV_w (5j) par RV_m (22j).")
print(f"Impact : IC passe de ~0.45 à ~0.35 (-22%)")

### Biais 3 — Mesure de volatilité trop bruitée

Le modèle initial utilisait comme mesure de volatilité un simple $|r_{\text{daily}}|$ (un seul rendement par jour). C'est un proxy très bruité : une journée avec de larges oscillations intraday mais un rendement net proche de zéro serait classée "faible volatilité".

En passant à la **vraie RV intraday** (~350 observations/jour), on obtient une mesure beaucoup plus précise.

In [ ]:
# Comparaison visuelle : RV intraday vs proxy journalier
rv_intraday = df_demo['rv_intraday'].dropna()
rv_mad = np.abs(df_demo['returns']).reindex(rv_intraday.index)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(rv_intraday.index[-200:], rv_intraday.values[-200:] * 100,
             linewidth=1, label='RV intraday (350 obs/j)', color='#2c3e50')
axes[0].plot(rv_mad.index[-200:], rv_mad.values[-200:] * 100,
             linewidth=1, alpha=0.7, label='|rendement journalier| (1 obs/j)', color='#e74c3c')
axes[0].set_ylabel('Volatilité (%)')
axes[0].set_title('200 derniers jours (AAPL)')
axes[0].legend(fontsize=8)

v = rv_intraday.index.intersection(rv_mad.dropna().index)
rho, _ = spearmanr(rv_intraday.loc[v], rv_mad.loc[v])
axes[1].scatter(rv_intraday.loc[v] * 100, rv_mad.loc[v] * 100, alpha=0.2, s=5, color='#3498db')
axes[1].set_xlabel('RV intraday (%)')
axes[1].set_ylabel('|Rendement journalier| (%)')
axes[1].set_title(f'Le proxy est très dispersé (ρ = {rho:.3f})')
axes[1].plot([0, 5], [0, 5], 'r--', linewidth=0.8, alpha=0.5)

plt.tight_layout()
plt.show()

print(f"Correction : passer à la RV intraday (√Σr²_1min) pour les features ET la cible.")
print(f"Impact : IC passe de ~0.35 à 0.291 (-17%)")

### Bilan : 52% du signal était artificiel

Après avoir corrigé les 3 biais, l'IC tombe de **0.61 à 0.29**. Ce n'est pas une dégradation du modèle — c'est une mesure honnête de sa performance réelle. Un IC de 0.29 est d'ailleurs un bon résultat (le modèle classe correctement les jours ~60% du temps).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

labels = ['Modèle initial\n(biaisé)', '+ Purging\n(gap=4)', '+ Découplage\n(denom=RV_m)', 'Après corrections\n(baseline honnête)']
ics = [0.609, 0.45, 0.35, 0.291]
colors = ['#e74c3c', '#e67e22', '#f39c12', '#27ae60']

bars = ax.bar(labels, ics, color=colors, edgecolor='white', width=0.6)
bars[0].set_hatch('///')
bars[0].set_edgecolor('#c0392b')

for bar, ic in zip(bars, ics):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{ic:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

for i in range(1, len(ics)):
    delta = (ics[i] - ics[i-1]) / ics[i-1] * 100
    ax.annotate(f'{delta:+.0f}%', xy=(i, ics[i] + 0.02),
                fontsize=9, ha='center', color='#c0392b', fontweight='bold')

ax.set_ylabel('Information Coefficient (Spearman)')
ax.set_title('Diagnostic méthodologique — chaque correction retire du signal artificiel')
ax.set_ylim(0, 0.75)
ax.axhline(0.291, color='green', linestyle='--', alpha=0.5, linewidth=0.8)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

print("Leçon : en finance quantitative, toujours vérifier ses métriques")
print("avant de chercher à améliorer le modèle.")

---
## 4. Amélioration 1 — Caractériser la *nature* de la volatilité

**Raisonnement :** Deux journées peuvent avoir la même RV mais des implications très différentes :
- Un **flash crash** (asymétrie forte, un seul gros mouvement) → la vol tend à persister
- Des **oscillations symétriques** (distribution normale) → la vol tend à revenir à la moyenne

Les features de base ne capturent que le *niveau* de la volatilité. On a besoin d'informer le modèle sur sa *nature*.

**3 features ajoutées :**
- **RSkew** (asymétrie réalisée) : négatif = crash risk
- **RKurt** (kurtosis réalisé) : > 3 = queues épaisses
- **Jump_ratio** : proportion de la variance due à des sauts discrets (via la bipower variation)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

df_demo = all_data['AAPL']

if 'rskew_intraday' in df_demo.columns:
    rskew = df_demo['rskew_intraday'].rolling(5).mean().dropna()
    axes[0].plot(rskew.index, rskew.values, linewidth=0.6, color='#8e44ad')
    axes[0].axhline(0, color='gray', linestyle='--', linewidth=0.5)
    axes[0].set_ylabel('RSkew (5j)')
    axes[0].set_title('Asymétrie réalisée — négatif = les mouvements baissiers dominent')
    axes[0].fill_between(rskew.index, rskew.values, 0,
                         where=rskew.values < 0, alpha=0.3, color='red', label='Crash risk')
    axes[0].fill_between(rskew.index, rskew.values, 0,
                         where=rskew.values >= 0, alpha=0.3, color='green', label='Rallye')
    axes[0].legend(fontsize=8)

if 'rkurt_intraday' in df_demo.columns:
    rkurt = df_demo['rkurt_intraday'].rolling(5).mean().dropna()
    axes[1].plot(rkurt.index, rkurt.values, linewidth=0.6, color='#d35400')
    axes[1].axhline(3, color='gray', linestyle='--', linewidth=0.5, label='Gaussien = 3')
    axes[1].set_ylabel('RKurt (5j)')
    axes[1].set_title('Kurtosis réalisé — au-dessus de 3 = queues plus épaisses que la normale')
    axes[1].legend(fontsize=8)

if 'bpv_intraday' in df_demo.columns and 'rv_intraday' in df_demo.columns:
    rv_sq = df_demo['rv_intraday'] ** 2
    bpv = df_demo['bpv_intraday']
    jr = (np.maximum(0, rv_sq - bpv) / rv_sq.replace(0, np.nan)).rolling(5).mean().dropna()
    axes[2].plot(jr.index, jr.values, linewidth=0.6, color='#c0392b')
    axes[2].set_ylabel('Jump ratio (5j)')
    axes[2].set_title('Proportion de la vol due aux sauts — pics = événements discrets')
    axes[2].set_ylim(0, min(1, jr.quantile(0.99) * 1.5))

plt.tight_layout()
plt.show()

print("Résultat : IC passe de 0.291 à 0.310 (+6.5%)")
print("Gain modeste mais systématique — la nature de la vol contient bien de l'info prédictive.")

---
## 5. Amélioration 2 — Adapter la fenêtre d'entraînement au régime de marché

**Raisonnement :** Jusqu'ici on entraîne le modèle sur une fenêtre fixe de 252 jours (1 an). Mais :
- En marché **calme**, les relations sont stables → on a intérêt à utiliser **plus** de données (2 ans)
- En **crise**, le marché a changé de régime → les données anciennes ne sont plus pertinentes → fenêtre **courte** (9 mois)

Le VIX (volatilité implicite du S&P 500) est un indicateur naturel de régime. On définit la taille de la fenêtre par interpolation linéaire :

| Régime | VIX | Fenêtre |
|--------|-----|--------|
| Calme | ≤ 18 | 504 jours (~2 ans) |
| Transition | 18-22 | interpolation linéaire |
| Crise | ≥ 22 | 189 jours (~9 mois) |

In [ ]:
vix_range = np.linspace(10, 40, 300)
windows = [adaptive_train_window(v) for v in vix_range]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# La fonction
axes[0].plot(vix_range, windows, color='#2c3e50', linewidth=2)
axes[0].fill_between(vix_range, windows, alpha=0.1, color='#3498db')
axes[0].scatter([18, 22], [504, 189], color='#e74c3c', s=80, zorder=5)
axes[0].annotate('VIX=18 → 504j', xy=(18, 504), xytext=(24, 480), fontsize=9,
                 arrowprops=dict(arrowstyle='->', color='gray'))
axes[0].annotate('VIX=22 → 189j', xy=(22, 189), xytext=(28, 220), fontsize=9,
                 arrowprops=dict(arrowstyle='->', color='gray'))
axes[0].axvspan(10, 18, alpha=0.05, color='green')
axes[0].axvspan(22, 40, alpha=0.05, color='red')
axes[0].set_xlabel('VIX')
axes[0].set_ylabel('Fenêtre (jours)')
axes[0].set_title('Règle : VIX → taille de la fenêtre d\'entraînement')
axes[0].set_xlim(10, 40)

# Dans le temps
vix_daily = vix.dropna()
w_daily = [adaptive_train_window(v) for v in vix_daily.values]
axes[1].plot(vix_daily.index, w_daily, linewidth=0.8, color='#2c3e50')
axes[1].axhline(252, color='red', linestyle='--', linewidth=0.8, label='Fenêtre fixe 252j')
axes[1].fill_between(vix_daily.index, w_daily, 252,
                      where=[w > 252 for w in w_daily], alpha=0.2, color='green', label='Plus de données (calme)')
axes[1].fill_between(vix_daily.index, w_daily, 252,
                      where=[w < 252 for w in w_daily], alpha=0.2, color='red', label='Moins de données (crise)')
axes[1].set_ylabel('Fenêtre (jours)')
axes[1].set_title('Évolution de la fenêtre dans le temps')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

print("Résultat (comparaison équitable, même période d'évaluation) :")
print("  Fenêtre fixe 252j : IC = 0.305")
print("  Fenêtre adaptative : IC = 0.331 (+8.7%)")
print("  Gain positif sur 4/5 stocks.")

In [ ]:
# Le VIX au fil du temps — quand est-ce que l'adaptation agit ?
fig, ax = plt.subplots(figsize=(12, 4))

ax.plot(vix.index, vix.values, linewidth=0.8, color='#2c3e50')
ax.axhspan(0, 18, alpha=0.1, color='green', label='Calme → fenêtre longue (504j)')
ax.axhspan(18, 22, alpha=0.1, color='orange', label='Transition')
ax.axhspan(22, vix.max() + 5, alpha=0.1, color='red', label='Crise → fenêtre courte (189j)')
ax.set_ylabel('VIX')
ax.set_title('Régimes de marché et conséquences sur la fenêtre d\'entraînement')
ax.legend(loc='upper right', fontsize=8)
ax.set_ylim(10, min(vix.max() + 5, 60))

pct_calme = (vix <= 18).mean() * 100
pct_crise = (vix >= 22).mean() * 100
ax.text(0.02, 0.95, f'{pct_calme:.0f}% du temps en calme, {pct_crise:.0f}% en crise',
        transform=ax.transAxes, va='top', fontsize=9,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

---
## 6. Amélioration 3 — L'effet de levier (et la plus grosse surprise)

**Raisonnement :** Toutes nos features mesurent des *magnitudes* (volatilité, volumes, asymétrie, kurtosis...). Aucune ne dit si le marché a **monté ou baissé**. Or il existe un fait empirique très robuste en finance :

> **L'effet de levier** (Black 1976) : les baisses de prix provoquent des hausses de volatilité **plus fortes** que les hausses de même amplitude.

Deux mécanismes :
1. **Mécanique** : quand le prix baisse, le ratio dette/capitalisation augmente → risque perçu en hausse → vol en hausse
2. **Comportemental** : les baisses déclenchent des stop-loss et du panic selling → amplification de la vol

**2 features ajoutées :**
- **Ret_w** : rendement cumulé 5 jours (la *direction* du marché)
- **Leverage_22d** : corrélation rolling 22j entre rendement et ΔRV (la *force* de l'effet de levier)

In [ ]:
df_demo = all_data['AAPL']
returns = df_demo['returns']
rv = df_demo['rv_intraday'] if 'rv_intraday' in df_demo.columns else np.abs(returns)

ret_w = returns.rolling(5).sum()
delta_rv = rv.diff()
leverage_22d = returns.rolling(22).corr(delta_rv)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# L'effet de levier brut
v = returns.notna() & delta_rv.notna()
axes[0, 0].scatter(returns[v].values * 100, delta_rv[v].values * 100, alpha=0.1, s=3, color='#2c3e50')
axes[0, 0].axhline(0, color='gray', linestyle='--', linewidth=0.5)
axes[0, 0].axvline(0, color='gray', linestyle='--', linewidth=0.5)
axes[0, 0].set_xlabel('Rendement (%)')
axes[0, 0].set_ylabel('ΔRV (%)')
axes[0, 0].set_title('Rendement vs ΔRV — l\'asymétrie est visible à l\'œil nu')

# Leverage_22d dans le temps
lev = leverage_22d.dropna()
axes[0, 1].plot(lev.index, lev.values, linewidth=0.5, color='#8e44ad')
axes[0, 1].axhline(0, color='gray', linestyle='--', linewidth=0.5)
axes[0, 1].axhline(-0.3, color='red', linestyle=':', linewidth=0.8, label='Leverage fort')
axes[0, 1].fill_between(lev.index, lev.values, 0, where=lev.values < 0, alpha=0.2, color='red')
axes[0, 1].set_ylabel('Corr(ret, ΔRV)')
axes[0, 1].set_title('Leverage_22d — la force de l\'effet varie beaucoup dans le temps')
axes[0, 1].legend(fontsize=8)

# Distribution
axes[1, 0].hist(lev.values, bins=60, color='#8e44ad', edgecolor='white', alpha=0.8)
axes[1, 0].axvline(lev.mean(), color='red', linestyle='--', label=f'μ = {lev.mean():.3f}')
axes[1, 0].axvline(0, color='gray', linestyle='--', linewidth=0.5)
axes[1, 0].set_xlabel('Leverage_22d')
axes[1, 0].set_title('Moyenne négative → l\'effet de levier est présent en général')
axes[1, 0].legend(fontsize=8)

# Ret_w
rw = ret_w.dropna()
axes[1, 1].plot(rw.index, rw.values * 100, linewidth=0.5, color='#27ae60')
axes[1, 1].axhline(0, color='gray', linestyle='--', linewidth=0.5)
axes[1, 1].fill_between(rw.index, rw.values * 100, 0,
                         where=rw.values < 0, alpha=0.2, color='red', label='Baisse')
axes[1, 1].fill_between(rw.index, rw.values * 100, 0,
                         where=rw.values >= 0, alpha=0.2, color='green', label='Hausse')
axes[1, 1].set_ylabel('Rendement cumulé 5j (%)')
axes[1, 1].set_title('Ret_w — la seule feature directionnelle du modèle')
axes[1, 1].legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Résultat
stocks = ['AAPL', 'MSFT', 'NVDA', 'JPM', 'META']
ic_before = [0.287, 0.359, 0.280, 0.312, 0.419]
ic_after = [0.382, 0.427, 0.303, 0.371, 0.486]

x = np.arange(len(stocks))
w = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
b1 = ax.bar(x - w/2, ic_before, w, label='Sans leverage (18 features)', color='#3498db', edgecolor='white')
b2 = ax.bar(x + w/2, ic_after, w, label='Avec leverage (20 features)', color='#2c3e50', edgecolor='white')

for bar, ic in zip(list(b1) + list(b2), ic_before + ic_after):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{ic:.3f}', ha='center', va='bottom', fontsize=8)

for i in range(len(stocks)):
    delta = (ic_after[i] - ic_before[i]) / ic_before[i] * 100
    ax.annotate(f'+{delta:.0f}%', xy=(i + w/2, ic_after[i] + 0.02),
                fontsize=9, ha='center', color='#27ae60', fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(stocks)
ax.set_ylabel('IC (Spearman)')
ax.set_title('Impact des features leverage — gain systématique sur 5/5 stocks (+18.9% en moyenne)')
ax.set_ylim(0, 0.58)
ax.legend(loc='upper left')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

---
## 7. La surprise — une feature « invisible » porte tout le gain

Pour comprendre laquelle des deux features leverage est responsable du gain, on fait une **étude d'ablation** : tester chacune séparément.

In [ ]:
# Ablation
configs = ['Sans leverage\n(18 features)', '+ Ret_w\nseul', '+ Leverage_22d\nseul', 'Les deux\n(20 features)']
ics_abl = [0.331, 0.334, 0.397, 0.394]
colors_abl = ['#3498db', '#5dade2', '#1a5276', '#2c3e50']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(configs, ics_abl, color=colors_abl, edgecolor='white', width=0.6)

for bar, ic in zip(bars, ics_abl):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.004,
            f'{ic:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.axhline(0.331, color='gray', linestyle='--', alpha=0.5, linewidth=0.7)
ax.annotate('Ret_w : +0.8%\n(quasi rien)', xy=(1, 0.334), xytext=(1, 0.36),
            fontsize=9, ha='center', color='#7f8c8d',
            arrowprops=dict(arrowstyle='->', color='gray'))
ax.annotate('Leverage_22d : +19.7%\n(presque tout le gain !)', xy=(2, 0.397), xytext=(2, 0.43),
            fontsize=9, ha='center', color='#e74c3c', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='#e74c3c'))

ax.set_ylabel('IC moyen (5 stocks)')
ax.set_title('Étude d\'ablation — Leverage_22d porte 97% du gain')
ax.set_ylim(0, 0.48)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

### Pourquoi est-ce surprenant ?

Leverage_22d n'a **aucune corrélation significative** avec la cible (ρ = -0.017, p = 0.55). Si on faisait une sélection de features classique (corrélation marginale), on l'éliminerait. Un modèle linéaire l'ignorerait complètement.

Pourtant elle porte **97% du gain**. Comment est-ce possible ?

In [ ]:
# Montrer visuellement le paradoxe
features_demo = model.create_features(df_demo, vix, cs_features.get('AAPL'))
target_demo = model.create_target(df_demo)

v = ~(features_demo.isna().any(axis=1) | np.isnan(target_demo))
X = features_demo[v]
y = target_demo[v]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. Corrélations marginales
correlations = {col: spearmanr(X[col].values, y)[0] for col in X.columns}
sc = dict(sorted(correlations.items(), key=lambda x: abs(x[1]), reverse=True))
cb = ['#e74c3c' if k in ['Leverage_22d', 'Ret_w'] else '#3498db' for k in sc.keys()]

axes[0].barh(range(len(sc)), list(sc.values()), color=cb)
axes[0].set_yticks(range(len(sc)))
axes[0].set_yticklabels(list(sc.keys()), fontsize=7)
axes[0].set_xlabel('ρ Spearman avec la cible')
axes[0].set_title('Corrélation marginale\n(rouge = features leverage)')
axes[0].invert_yaxis()

# 2. Leverage_22d vs cible : rien !
rho_lev, p_lev = spearmanr(X['Leverage_22d'].values, y)
axes[1].scatter(X['Leverage_22d'].values, y, alpha=0.1, s=3, color='#8e44ad')
axes[1].axhline(0, color='gray', linestyle='--', linewidth=0.5)
axes[1].set_xlabel('Leverage_22d')
axes[1].set_ylabel('Cible')
axes[1].set_title(f'Leverage_22d vs cible\nρ = {rho_lev:.3f}, p = {p_lev:.2f}\n→ Aucun signal linéaire !')

# 3. Ret_w vs cible : signal... mais redondant
rho_ret, _ = spearmanr(X['Ret_w'].values, y)
axes[2].scatter(X['Ret_w'].values, y, alpha=0.1, s=3, color='#27ae60')
axes[2].axhline(0, color='gray', linestyle='--', linewidth=0.5)
axes[2].set_xlabel('Ret_w')
axes[2].set_ylabel('Cible')
axes[2].set_title(f'Ret_w vs cible\nρ = {rho_ret:.3f} — signal ok\nmais redondant avec RV_neg_w')

plt.tight_layout()
plt.show()

rho_redun, _ = spearmanr(X['Ret_w'].values, X['RV_neg_w'].values)
print(f"\nRet_w ↔ RV_neg_w : ρ = {rho_redun:.3f} (l'info directionnelle est déjà dans le modèle)")
print(f"Leverage_22d ↔ cible : ρ = {rho_lev:.3f} (non significatif)")
print(f"\nEt pourtant Leverage_22d apporte +19.7% d'IC. Comment ?")

### L'explication : Leverage_22d est un « conditionneur »

Leverage_22d ne prédit pas la cible directement. Elle **modifie la relation entre les autres features et la cible**. Concrètement, les arbres de XGBoost créent des règles conditionnelles comme :

```
SI Leverage_22d < -0.3 (effet de levier actif)
    ET RV_w > seuil (vol élevée)
    → ALORS la vol va persister (cible élevée)

SI Leverage_22d > 0 (pas d'effet de levier)
    ET RV_w > seuil (vol élevée)
    → ALORS la vol va mean-reverter plus vite (cible modérée)
```

La feature ne « prédit » rien seule, mais elle **conditionne la prédiction** des autres features. C'est un type d'interaction que seul un modèle non linéaire peut exploiter.

**Leçon méthodologique importante :** en sélection de features pour les modèles non linéaires (XGBoost, Random Forest...), la corrélation marginale avec la cible est **trompeuse**. Des features à ρ ≈ 0 peuvent être cruciales via leurs interactions. Seule l'évaluation end-to-end (ajouter la feature et mesurer l'impact) est fiable.

---
## 8. Vue d'ensemble

Récapitulatif du chemin parcouru :

In [ ]:
results = pd.DataFrame({
    'Étape': [
        'Baseline honnête (après corrections)',
        '+ Moments supérieurs (RSkew, RKurt, Jump_ratio)',
        '+ Fenêtre adaptative VIX',
        '+ Effet de levier (Ret_w, Leverage_22d)'
    ],
    'Raisonnement': [
        'Corriger 3 biais : purging, découplage, RV intraday',
        'Informer le modèle sur la *nature* de la vol',
        'Adapter au régime de marché (calme vs crise)',
        'Capturer l\'asymétrie rendement → vol'
    ],
    'IC': [0.291, 0.310, 0.331, 0.394],
    'Δ relatif': ['—', '+6.5%', '+8.7%', '+18.9%'],
    'Δ cumulatif': ['—', '+6.5%', '+13.7%', '+35.4%'],
})

results.set_index('Étape')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

all_labels = ['Initial\n(biaisé)', 'Après\ncorrections', '+ Moments\nsupérieurs', '+ Fenêtre\nadaptative', '+ Effet de\nlevier']
all_ics = [0.609, 0.291, 0.310, 0.331, 0.394]
all_colors = ['#e74c3c', '#bdc3c7', '#95a5a6', '#3498db', '#2c3e50']

bars = ax.bar(all_labels, all_ics, color=all_colors, edgecolor='white', width=0.6)
bars[0].set_hatch('///')
bars[0].set_edgecolor('#c0392b')

for bar, ic in zip(bars, all_ics):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{ic:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.annotate('', xy=(1, 0.30), xytext=(0, 0.60),
            arrowprops=dict(arrowstyle='->', color='#e74c3c', lw=2))
ax.text(0.5, 0.48, '-52%\nbiais retirés', ha='center', fontsize=10,
        color='#e74c3c', fontweight='bold')

ax.annotate('', xy=(4, 0.40), xytext=(1, 0.30),
            arrowprops=dict(arrowstyle='->', color='#27ae60', lw=2))
ax.text(2.5, 0.25, '+35.4%\nsignal réel', ha='center', fontsize=10,
        color='#27ae60', fontweight='bold')

ax.set_ylabel('Information Coefficient (Spearman)')
ax.set_title('Le chemin complet : diagnostic → corrections → améliorations')
ax.set_ylim(0, 0.72)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

---
## Ce qu'on retient

**1. Toujours vérifier ses métriques.**  
52% du signal initial était artificiel. Si on avait directement cherché à améliorer le modèle sans diagnostiquer les biais, on aurait construit sur du sable.

**2. La nature de l'information compte plus que sa quantité.**  
5 features supplémentaires (15 → 20) ont suffi pour +35.4% d'IC. Chacune apporte une dimension *qualitativement* différente : forme de la distribution, régime de marché, asymétrie rendement-vol.

**3. Les modèles non linéaires voient des choses que les modèles linéaires ne voient pas.**  
Leverage_22d (ρ = -0.017 avec la cible) porte 97% du gain des features leverage. Un modèle linéaire l'aurait rejeté. L'information prédictive réside dans les *interactions entre features*, pas dans les corrélations marginales.

**4. L'adaptation au contexte est payante.**  
Plutôt qu'une fenêtre d'entraînement fixe, on ajuste en temps réel au régime de marché. Le même principe pourrait s'appliquer à d'autres hyperparamètres.

---
*Notebook complémentaire au papier : "Prédiction de la Volatilité Réalisée par Modèle HAR-RV Étendu avec XGBoost"*